# 02 · Baseline evaluation (H1–H3)

First notebook in which models see the evaluation set. Everything here follows the frozen pre-registration (section 5).

- **Part A (checks, no evaluation items):** verify the data files are the frozen ones, read the Hausa prompt, and run a smoke test on 10 **training-pool** items to confirm model access and answer parsing.
- **Stop and check** the smoke-test output before Part B.
- **Part B:** 3 models × 7 conditions × 500 items, then the pre-registered H1–H3 decisions and sensitivity analyses.

Use an **A100 or L4** runtime (bf16). Every model output is cached on Drive, so after a disconnect just re-run.

## 0 · Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
REPO_DIR = "/content/hausa-med-qa"
import os
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/avvas200/hausa-med-qa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git log -1 --format="%h %s"
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
/content/hausa-med-qa
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.0 MB/s eta 0:00:00
cd35807 Deviation 2: letter-probability scoring
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Llama 3.2 and MedGemma are gated: accept their licenses on huggingface.co first.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

In [3]:
import json, time, platform, gc
from collections import Counter
import numpy as np, pandas as pd, torch, transformers
from src import config as C, data as D, evaluate as E, stats as S
C.ensure_dirs()
if not torch.cuda.is_bf16_supported():
    print("WARNING: this GPU has no bf16 (e.g. T4). Switch to an A100 or L4 runtime; Gemma models are unreliable in fp16.")
print("torch", torch.__version__, "| transformers", transformers.__version__, "|", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | transformers 5.16.1 | Tesla T4


# Part A · Checks
## A1 · Are these the frozen data files?

In [4]:
manifest = json.load(open("results/data_manifest.json"))
for f in ["eval_en.jsonl", "eval_ha.jsonl", "eval_ha2en.jsonl"]:
    ok = D.sha256(C.DATA / f) == manifest["sha256"][f]
    print(f"{f:20s}", "OK" if ok else "MISMATCH")
    assert ok, f"{f} differs from the frozen manifest - stop and check before evaluating"
data = {f: D.read_jsonl(C.DATA / f) for f in ["eval_en.jsonl", "eval_ha.jsonl", "eval_ha2en.jsonl"]}
ids = [r["id"] for r in data["eval_en.jsonl"]]
assert all([r["id"] for r in v] == ids for v in data.values()), "item order differs between files"
print(len(ids), "items, same order in all three files")

eval_en.jsonl        OK
eval_ha.jsonl        OK
eval_ha2en.jsonl     OK
500 items, same order in all three files


## A2 · Read the prompts (training-pool example)
Read the **Hausa** instruction carefully. The pre-registration says it is hand-corrected by you. If anything should change, edit `EVAL_PROMPTS["ha"]` in `src/config.py`, commit, restart, and re-run from the top. Only the instruction lines matter; the question and options are filled in automatically.

In [5]:
pool_ha = D.read_jsonl(C.DATA / "train_pool_ha.jsonl")
pool_en = D.read_jsonl(C.DATA / "train_pool_en.jsonl")
print(E.prompt_text(pool_en[0], pool_en[0]["perms"][0], "en"), "\n", "-" * 60)
print(E.prompt_text(pool_ha[0], pool_ha[0]["perms"][0], "ha"))

The following is a multiple-choice question about medicine. Choose the single best answer. Reply with only the letter (A, B, C or D).

Question: The interossei are supplied by
A. Superficial palmar branch of radial artery
B. Deep palmar branch of ulnar artery
C. Superficial palmar arch
D. Deep palmar arch
Answer: 
 ------------------------------------------------------------
Ga tambaya mai zaɓi daga cikin amsoshi game da ilimin likitanci. Zaɓi amsa ɗaya mafi dacewa. Ka amsa da harafi kawai (A, B, C ko D).

Tambaya: Ana samar da interossei ta hanyar
A. Superficial palmar branch of radial artery
B. Deep palmar branch of ulnar artery
C. Superficial palmar arch
D. Deep palmar arch
Amsa:


## A3 · Smoke test (10 training-pool items per model, EN and HA)
Confirms that each model loads, and shows its raw answers next to the parsed letter. The answer turn is prefilled with `Answer:` / `Amsa:` (deviation 1). If MedGemma can't be accessed, the pre-registered substitute (Gemma-3-4B-IT) is used and recorded.

In [6]:
def load_resolved(key):
    name = C.EVAL_MODELS[key]
    try:
        return name, *E.load_model(name)
    except OSError as e:
        if key == "medgemma" and any(s in str(e).lower() for s in ("gated", "401", "403", "access")):
            print(f"MedGemma not accessible ({str(e).splitlines()[0][:120]}) -> using {C.MEDGEMMA_SUBSTITUTE}")
            return C.MEDGEMMA_SUBSTITUTE, *E.load_model(C.MEDGEMMA_SUBSTITUTE)
        raise

resolved = {}
smoke = [r for r in pool_ha if r["question"]][:10]
smoke_en = {r["id"]: r for r in pool_en}
for key in C.EVAL_MODELS:
    name, tok, model, dtype = load_resolved(key)
    resolved[key] = {"model": name, "dtype": dtype}
    print(f"\n===== {key}: {name} ({dtype}) =====")
    for lang, recs in (("en", [smoke_en[r["id"]] for r in smoke]), ("ha", smoke)):
        rows = E.run_condition(tok, model, recs, lang, 0, C.OUTPUTS / "smoke_v3" / key / f"{lang}.jsonl")
        parsed = sum(r["pos"] >= 0 for r in rows)
        agree = sum(r["pos"] == r["pos_lp"] for r in rows if r["pos"] >= 0)
        print(f"[{lang}] parsed {parsed}/10 | letter-prob agrees with parsed letter {agree}/{parsed} | raw:",
              [r["raw"] for r in rows[:5]])
    del model, tok; gc.collect(); torch.cuda.empty_cache()
json.dump(resolved, open(C.RESULTS / "eval_models_resolved.json", "w"), indent=2)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]


===== medgemma: google/medgemma-4b-it (bfloat16) =====
[en] parsed 10/10 | letter-prob agrees with parsed letter 10/10 | raw: ['B', 'B', 'A', 'C', 'B']
[ha] parsed 10/10 | letter-prob agrees with parsed letter 10/10 | raw: ['D', 'B', 'C', 'C', 'C']


**Stop here.** Paste the smoke-test output into the chat. Continue only if every model loaded and parsing works. If you change anything (prompt or parser), commit it, restart, and delete the `outputs/smoke` folder on Drive before re-running.

# Part B · Evaluation
## B1 · Run all models and conditions (resumable)

In [7]:
resolved = json.load(open(C.RESULTS / "eval_models_resolved.json"))
run_log = {"prefill": C.EVAL_PREFILL, "started": time.strftime("%Y-%m-%d %H:%M:%S"), "models": resolved,
           "commit": os.popen("git rev-parse --short HEAD").read().strip(),
           "gpu": torch.cuda.get_device_name(0), "batch": C.EVAL_BATCH,
           "max_new_tokens": C.EVAL_MAX_NEW_TOKENS,
           "versions": {"python": platform.python_version(), "torch": torch.__version__,
                        "transformers": transformers.__version__}}
for key, info in resolved.items():
    tok, model, dtype = E.load_model(info["model"])
    print(f"\n===== {key}: {info['model']} ({dtype}) =====")
    for cond, fname, lang, p, prefill in C.CONDITIONS:
        t0 = time.time()
        rows = E.run_condition(tok, model, data[fname], lang, p, C.OUTPUTS / key / f"{cond}.jsonl",
                               prefill=prefill)
        acc = np.mean([r["correct"] for r in rows]); pr = np.mean([r["pos"] >= 0 for r in rows])
        print(f"  {cond:16s} acc {acc:.3f} | parsed {pr:.3f} | {time.time() - t0:.0f}s")
    del model, tok; gc.collect(); torch.cuda.empty_cache()
run_log["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
json.dump(run_log, open(C.RESULTS / "eval_run_log.json", "w"), indent=2)
json.dump(run_log, open("results/eval_run_log.json", "w"), indent=2)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


===== llama: meta-llama/Llama-3.2-3B-Instruct (bfloat16) =====
  en_p0            acc 0.618 | parsed 1.000 | 248s
  en_p1            acc 0.612 | parsed 1.000 | 247s
  en_p2            acc 0.594 | parsed 1.000 | 248s
  ha_p0            acc 0.276 | parsed 1.000 | 433s
  ha_p1            acc 0.342 | parsed 1.000 | 434s
  ha_p2            acc 0.312 | parsed 1.000 | 434s
  ha2en_p0         acc 0.450 | parsed 1.000 | 247s
  en_p0_noprefill  acc 0.598 | parsed 0.994 | 255s
  ha_p0_noprefill  acc 0.136 | parsed 0.476 | 430s


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


===== qwen: Qwen/Qwen2.5-3B-Instruct (bfloat16) =====
  en_p0            acc 0.526 | parsed 1.000 | 240s
  en_p1            acc 0.526 | parsed 1.000 | 245s
  en_p2            acc 0.512 | parsed 0.998 | 241s
  ha_p0            acc 0.122 | parsed 0.470 | 414s
  ha_p1            acc 0.122 | parsed 0.432 | 413s
  ha_p2            acc 0.134 | parsed 0.462 | 413s
  ha2en_p0         acc 0.392 | parsed 1.000 | 239s
  en_p0_noprefill  acc 0.542 | parsed 1.000 | 239s
  ha_p0_noprefill  acc 0.110 | parsed 0.400 | 408s


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


===== medgemma: google/medgemma-4b-it (bfloat16) =====
  en_p0            acc 0.550 | parsed 1.000 | 250s
  en_p1            acc 0.560 | parsed 1.000 | 255s
  en_p2            acc 0.552 | parsed 1.000 | 254s
  ha_p0            acc 0.356 | parsed 1.000 | 368s
  ha_p1            acc 0.378 | parsed 1.000 | 370s
  ha_p2            acc 0.386 | parsed 1.000 | 369s
  ha2en_p0         acc 0.402 | parsed 1.000 | 256s
  en_p0_noprefill  acc 0.550 | parsed 1.000 | 252s
  ha_p0_noprefill  acc 0.348 | parsed 1.000 | 362s


## B2 · Pre-registered decisions (H1–H3) and sensitivity analyses

In [8]:
models = list(resolved)

def vec(m, cond, field):
    by = {r["id"]: r for r in D.read_jsonl(C.OUTPUTS / m / f"{cond}.jsonl")}
    assert set(by) == set(ids), f"{m}/{cond} is incomplete"
    return np.array([by[i][field] for i in ids])

corr = {m: {c[0]: vec(m, c[0], "correct").astype(bool) for c in C.CONDITIONS} for m in models}
chc = {m: {c[0]: vec(m, c[0], "choice") for c in C.CONDITIONS} for m in models}
pos = {m: {c[0]: vec(m, c[0], "pos") for c in C.CONDITIONS} for m in models}
# Secondary (deviation 2): letter-probability scoring from the same runs
corr_lp = {m: {c[0]: vec(m, c[0], "correct_lp").astype(bool) for c in C.CONDITIONS} for m in models}
chc_lp = {m: {c[0]: vec(m, c[0], "choice_lp") for c in C.CONDITIONS} for m in models}

def analyse(mask, corr=corr, chc=chc):
    sub = lambda d, c: {m: d[m][c][mask] for m in models}
    h1 = S.h1_language_gap(sub(corr, "en_p0"), sub(corr, "ha_p0"))
    h2 = S.h2_translate_test(sub(corr, "en_p0"), sub(corr, "ha_p0"), sub(corr, "ha2en_p0"),
                             {m: h1[m]["supported"] for m in models})
    three = lambda d, lang: {m: np.stack([d[m][f"{lang}_p{p}"] for p in range(3)], 1)[mask] for m in models}
    h3 = S.h3_consistency_gap(three(chc, "en"), three(chc, "ha"))
    robust = {m: {lang: float(S.robust_correct_vector(three(corr, lang)[m]).mean()) for lang in ("en", "ha")}
              for m in models}
    return {"n_items": int(mask.sum()), "H1": h1, "H2": h2, "H3": h3, "robust_accuracy": robust}

flagged = set(json.load(open(C.DATA / "rater_flagged_ids.json")))
fallback = set(json.load(open(C.DATA / "fallback_item_ids.json")))
masks = {"primary": np.ones(len(ids), bool),
         "sens_a_no_rater_flagged": np.array([i not in flagged for i in ids]),
         "sens_b_no_english_options": np.array([i not in fallback for i in ids])}
results = {k: analyse(v) for k, v in masks.items()}
results["secondary_letter_prob"] = analyse(masks["primary"], corr_lp, chc_lp)
results["accuracy_letter_prob"] = {m: {c: float(corr_lp[m][c].mean()) for c in corr_lp[m]} for m in models}

gold_pos = {p: Counter(r["perms"][p].index(r["answer_idx"]) for r in data["eval_en.jsonl"]) for p in range(3)}
results["majority_letter_baseline"] = {f"p{p}": max(c.values()) / len(ids) for p, c in gold_pos.items()}
results["parse_rate"] = {m: {c: float((pos[m][c] >= 0).mean()) for c in pos[m]} for m in models}
results["accuracy"] = {m: {c: float(corr[m][c].mean()) for c in corr[m]} for m in models}
results["models"] = resolved

def to_json(o):
    return o.item() if hasattr(o, "item") else str(o)
for path in ("results/baseline_results.json", C.RESULTS / "baseline_results.json"):
    json.dump(results, open(path, "w"), indent=2, default=to_json)

r = results["primary"]
print("Majority-letter baseline:", {k: round(v, 3) for k, v in results["majority_letter_baseline"].items()})
print(pd.DataFrame({m: {"EN": r["H1"][m]["acc_a"], "HA": r["H1"][m]["acc_b"],
                        "HA->EN": results["accuracy"][m]["ha2en_p0"],
                        "H1 gap": r["H1"][m]["diff"], "H1 p_holm": r["H1"][m]["p_holm"],
                        "H1": r["H1"][m]["supported"],
                        "H2 recovery": r["H2"].get(m, {}).get("recovery", {}).get("point", np.nan),
                        "H2": r["H2"].get(m, {}).get("supported", "not tested"),
                        "cons EN": r["H3"][m]["cons_en"], "cons HA": r["H3"][m]["cons_ha"],
                        "H3": r["H3"][m]["supported"]} for m in models}).T.to_string(float_format=lambda x: f"{x:.3f}"))
for k in ("sens_a_no_rater_flagged", "sens_b_no_english_options"):
    s = results[k]
    print(f"\n{k} (n={s['n_items']}):",
          {m: (round(float(s['H1'][m]['diff']), 3), s['H1'][m]['supported'], s['H3'][m]['supported']) for m in models})
s = results["secondary_letter_prob"]
print("\nSecondary, letter-probability scoring (not confirmatory):")
print(pd.DataFrame({m: {"EN": s["H1"][m]["acc_a"], "HA": s["H1"][m]["acc_b"],
                        "HA->EN": results["accuracy_letter_prob"][m]["ha2en_p0"],
                        "gap": s["H1"][m]["diff"], "cons EN": s["H3"][m]["cons_en"],
                        "cons HA": s["H3"][m]["cons_ha"]} for m in models}).T.to_string(float_format=lambda x: f"{x:.3f}"))
print("\nParse rates:", pd.DataFrame(results["parse_rate"]).round(3).to_string())

Majority-letter baseline: {'p0': 0.318, 'p1': 0.294, 'p2': 0.276}
            EN    HA HA->EN H1 gap H1 p_holm    H1 H2 recovery     H2 cons EN cons HA    H3
llama    0.618 0.276  0.450  0.342     0.000  True       0.509   True   0.670   0.210  True
qwen     0.526 0.122  0.392  0.404     0.000  True       0.668   True   0.676   0.084  True
medgemma 0.550 0.356  0.402  0.194     0.000  True       0.237  False   0.578   0.408  True

sens_a_no_rater_flagged (n=499): {'llama': (0.343, True, True), 'qwen': (0.405, True, True), 'medgemma': (0.194, True, True)}

sens_b_no_english_options (n=405): {'llama': (0.346, True, True), 'qwen': (0.405, True, True), 'medgemma': (0.222, True, True)}

Secondary, letter-probability scoring (not confirmatory):
            EN    HA  HA->EN   gap  cons EN  cons HA
llama    0.620 0.274   0.450 0.346    0.670    0.202
qwen     0.536 0.266   0.398 0.270    0.660    0.166
medgemma 0.550 0.360   0.400 0.190    0.576    0.408

Parse rates:                  llama   

## Next
Paste the B2 output into the chat. Then commit `results/baseline_results.json` and `results/eval_run_log.json`. Raw outputs stay on Drive under `outputs/` (used later for E1 error analysis).